# Bài 7: Các độ đo khoảng cách (Distance Measures)

"Khoảng cách" trên lưới điểm ảnh không đơn giản như nghe có vẻ. Chúng ta xem xét ba loại khác nhau:

1. **Các metric khoảng cách điểm-tới-điểm** (Manhattan, bàn cờ và Euclid) &mdash; đo hai điểm ảnh cách nhau bao xa.
2. **Các bộ ước lượng độ dài đường cong** (Freeman và một bộ ước lượng có trọng số được tinh chỉnh) &mdash; đo độ dài của một *đường cong đã số hóa* (ví dụ đường biên của một đốm) từ mã chuỗi (chain code) của nó.
3. **Biến đổi khoảng cách chamfer (chamfer distance transform)** &mdash; một phép xấp xỉ nhanh trên toàn ảnh, gán cho mỗi điểm ảnh khoảng cách tới phần tử gần nhất của một tập cho trước (ví dụ tiền cảnh), tính bằng hai lượt quét mành (raster scan).

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Phần 1: Các metric khoảng cách điểm-tới-điểm

Cho hai điểm ảnh $p=(x_1,y_1)$ và $q=(x_2,y_2)$, đặt $d_x = x_2-x_1$ và $d_y=y_2-y_1$. Ba metric phổ biến:

- **Manhattan / khối phố (city-block) ($L_1$)**: khoảng cách khi bạn chỉ có thể di chuyển dọc theo các trục của lưới (như đi bộ qua các dãy phố).

    $D_4(p,q) = |d_x| + |d_y|$ 
- **Bàn cờ (chessboard) ($L_\infty$)**: số nước đi của quân vua trên bàn cờ (bước chéo được "miễn phí").

    $D_8(p,q) = \max(|d_x|,\,|d_y|)$ 
- **Euclid ($L_2$)**: khoảng cách đường thẳng thông thường.

    $D_E(p,q) = \sqrt{d_x^2+d_y^2}$

Tên gọi $D_4$ và $D_8$ xuất phát từ việc mỗi metric là độ dài đường đi ngắn nhất khi chỉ cho phép di chuyển liên thông 4 (theo trục) hoặc liên thông 8 (theo trục + chéo), mỗi bước một đơn vị.

In [ ]:
def manhattan(dx, dy):
    return np.abs(dx) + np.abs(dy)

def chessboard(dx, dy):
    return np.maximum(np.abs(dx), np.abs(dy))

def euclidean(dx, dy):
    return np.sqrt(dx**2 + dy**2)

### Trực quan hóa ba metric dưới dạng trường khoảng cách

Với mỗi điểm ảnh trên lưới, ta tính khoảng cách của nó tới điểm ảnh trung tâm theo từng metric và hiển thị dưới dạng ảnh. Các metric chỉ trùng nhau dọc theo các trục — ở mọi chỗ khác chúng đều khác nhau.

In [ ]:
size = 101
half = size // 2
yy, xx = np.mgrid[-half:half + 1, -half:half + 1]

fields = {
    'Manhattan ($D_4$)': manhattan(xx, yy),
    'Chessboard ($D_8$)': chessboard(xx, yy),
    'Euclidean': euclidean(xx, yy),
}

# Compute a single vmin/vmax to prevent each imshow() from auto-scaling 
# (Manhattan's max is 100, Euclidean's is ~71, chessboard's is 50).
vmax = max(field.max() for field in fields.values())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, field) in zip(axes, fields.items()):
    im = ax.imshow(field, cmap='viridis', vmin=0, vmax=vmax)
    ax.contour(field, levels=8, colors='white', linewidths=0.5)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

Các đường đồng mức cho thấy hình dạng đặc trưng của mỗi metric: khoảng cách Manhattan tạo thành hình thoi, khoảng cách bàn cờ tạo thành hình vuông, và khoảng cách Euclid tạo thành hình tròn — hình dạng mà hầu hết mọi người trực giác nghĩ đến khi nói "cách đều".

## Phần 2: Ước lượng độ dài đường cong từ mã chuỗi

Giờ giả sử ta muốn ước lượng độ dài của một *đường cong*, chứ không chỉ khoảng cách giữa hai điểm. Hãy tạo một ảnh thử nghiệm là hình tròn có bán kính đã biết ($60$) và chu vi đã biết ($120 \pi = 377$). Lưu ý rằng số điểm ảnh biên ($336$) là một ước lượng tồi cho chu vi, vì $336 \neq 377$.

In [ ]:
radius = 60
img = np.zeros((150, 150), dtype=np.uint8)
cv2.circle(img, (75, 75), radius, 255, 1)

contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
pts = contours[0][:, 0, :]

plt.imshow(img, cmap='gray')
plt.title(f'Digitized circle, radius={radius}, {len(pts)} boundary pixels')
plt.axis('off')
plt.show()

**Mã chuỗi Freeman (Freeman chain code)** mô tả một đường cong số dưới dạng chuỗi các bước đơn vị theo 8 hướng có thể (0–7, cách nhau 45°). Bốn trong tám hướng là *chẵn* (ngang/dọc, độ dài thực 1) và bốn hướng là *lẻ* (chéo, độ dài thực $\sqrt{2}$).

Nếu mã chuỗi của một đường cong có $N_e$ bước chẵn và $N_o$ bước lẻ, **ước lượng Freeman** cho độ dài của nó đơn giản là

$$L_{\text{Freeman}} = N_e + \sqrt{2}\,N_o$$

Nghe có vẻ đúng, nhưng nó *ước lượng thừa* một cách có hệ thống đối với các đường cong trơn: đường biên của một hình tròn thật, khi được số hóa, sẽ luân phiên qua nhiều bước "bậc thang" zig-zag ngắn cộng lại dài hơn độ dài cung thực.

Để khắc phục, các trọng số chính xác $(1, \sqrt{2})$ có thể được thay bằng các trọng số $(a, b)$ được điều chỉnh để cực tiểu sai số trung bình trên nhiều đường cong đã số hóa — một ý tưởng đôi khi gắn với phép hiệu chỉnh độ dài mã chuỗi của Kimura. Các giá trị thường được trích dẫn (hơi khác nhau tùy nguồn) xấp xỉ $a \approx 0.948$ và $b \approx 1.343$:

$$L_{\text{corrected}} = a\,N_e + b\,N_o$$

In [ ]:
steps = np.diff(np.vstack([pts, pts[:1]]), axis=0)  # step from each boundary pixel to the next
is_diagonal = (steps[:, 0] != 0) & (steps[:, 1] != 0)

N_o = int(np.sum(is_diagonal))    # odd (diagonal) steps
N_e = int(np.sum(~is_diagonal))   # even (axis-aligned) steps

true_length = 2 * np.pi * radius
L_freeman = N_e + np.sqrt(2) * N_o

a, b = 0.948, 1.343
L_corrected = a * N_e + b * N_o

print(f'N_e (even/axis steps)     = {N_e}')
print(f'N_o (odd/diagonal steps)  = {N_o}')
print(f'number of boundary pixels = {len(pts):.1f}')
print(f'true circumference        = {true_length:.1f}')
print(f'Freeman estimate          = {L_freeman:.1f}  (error {100*(L_freeman-true_length)/true_length:+.1f}%)')
print(f'Corrected estimate        = {L_corrected:.1f}  (error {100*(L_corrected-true_length)/true_length:+.1f}%)')

Các trọng số đã hiệu chỉnh đưa ước lượng tới gần chu vi thực hơn nhiều, mà không cần gì hơn ngoài việc đếm số bước chẵn và lẻ — hữu ích khi bạn muốn một ước lượng độ dài rẻ trực tiếp từ mã chuỗi.

## Phần 3: Biến đổi khoảng cách chamfer

Khác với các metric điểm-tới-điểm ở Phần 1, vốn đo khoảng cách giữa hai điểm ảnh cụ thể, một **biến đổi khoảng cách (distance transform)** gán nhãn cho *mọi* điểm ảnh của ảnh nhị phân bằng khoảng cách của nó tới phần tử gần nhất trong một tập điểm ảnh cho trước — thường là tiền cảnh, hoặc nền — hữu ích cho việc tạo khung xương (skeletonization), so khớp hình dạng và lập kế hoạch đường đi. Tính nó *chính xác* với khoảng cách Euclid đòi hỏi so sánh mọi điểm ảnh với mọi điểm ảnh trong tập đó (chậm), hoặc một thuật toán tinh vi hơn.

**Biến đổi khoảng cách chamfer** là một phép xấp xỉ nhanh: thay vì một phép tìm kiếm toàn cục, nó lan truyền các khoảng cách cục bộ trên ảnh chỉ trong hai lượt quét mành (từ trên-trái xuống dưới-phải, rồi từ dưới-phải lên trên-trái), cộng thêm một chi phí cố định nhỏ cho mỗi bước sang điểm lân cận. Dùng trọng số nguyên $a=3$ cho các lân cận theo trục và $b=4$ cho các lân cận chéo (rồi chia cho 3 ở cuối) cho ra **khoảng cách chamfer 3-4** cổ điển, một phép xấp xỉ tốt và rẻ cho khoảng cách Euclid — cùng ý tưởng với các trọng số mã chuỗi Freeman/hiệu chỉnh ở Phần 2, nhưng áp dụng cho mọi điểm ảnh thay vì chỉ đường biên.

`cv2.distanceTransform` của OpenCV với `DIST_L2` tính khoảng cách chamfer. Tham số `maskSize` quyết định chất lượng của phép xấp xỉ: mặt nạ lớn hơn chính xác hơn nhưng tốn nhiều phép tính hơn.

Hình minh họa này cho thấy một ví dụ nhỏ về thuật toán chamfer 3-4 hai lượt khi chạy, lan truyền khoảng cách ra ngoài từ một đốm hạt giống (các điểm ảnh màu đỏ, nơi khoảng cách tới chính nó bằng 0). Lượt 1 quét từ trên-trái xuống dưới-phải, nên nó chỉ có thể lấy giá trị từ những lân cận đã thăm — nó "đổ bóng" các khoảng cách đã biết xuống dưới và sang phải. Lượt 2 quét theo chiều ngược lại (từ dưới-phải lên trên-trái), đổ bóng lên trên và sang trái, lấp đầy mọi thứ mà lượt 1 không với tới.

<img src="../animations/chamfer_demo.gif" alt="Ảnh động minh họa biến đổi khoảng cách chamfer 3-4 hai lượt, lan truyền khoảng cách từ một đốm hạt giống qua lượt quét xuôi rồi lượt quét ngược" style="display: block; margin: 0 auto; max-width: 680px;">

In [ ]:
img = np.zeros((100, 100), dtype=np.uint8)
cv2.circle(img, (50, 50), 10, 255, -1)
img = 255 - img  # we will compute distance from each background pixel to the nearest foreground (nonzero) pixel

im_dist3 = cv2.distanceTransform(img, cv2.DIST_L2, maskSize = 3) # (3,4) chamfer algorithm 
im_dist5 = cv2.distanceTransform(img, cv2.DIST_L2, maskSize = 5) # more accurate algorithm

error = im_dist3 - im_dist5
print(f'max error  = {np.abs(error).max():.3f} pixels')
print(f'mean error = {np.abs(error).mean():.3f} pixels')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, field, title in zip(
    axes,
    [im_dist3, im_dist5, error],
    ['Chamfer (3-4) approx.', 'More accurate estimate', 'Difference'],
):
    im = ax.imshow(field, cmap='viridis' if title != 'Difference' else 'coolwarm')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

Biến đổi chamfer xấp xỉ biến đổi khoảng cách Euclid chính xác, được tính bằng hai lượt quét tuyến tính nhanh thay vì một phép tìm kiếm lân cận gần nhất tốn kém.

### Bài tập

1. Lặp lại thí nghiệm hình tròn với vài bán kính khác nhau (ví dụ 10, 20, 40, 80). Phần trăm sai số của ước lượng Freeman giữ gần như không đổi, hay giảm dần khi hình tròn lớn hơn? Tại sao lại như vậy?
2. Khoảng cách bàn cờ $D_8$ là độ dài đường đi ngắn nhất chính xác khi cho phép bước chéo với chi phí 1. Bạn có thể dựng một metric kiểu bàn cờ cải biên, dùng chi phí bước chéo là $\sqrt{2}$ thay vì 1, hoạt động giống như một phiên bản cục bộ thô của các bộ ước lượng độ dài mã chuỗi ở trên không? So sánh nó với khoảng cách Euclid thực cho một vài điểm.